# Identifying emerging technology: focus on Europe
## Leveraging NUTS levels

## Prerrequisites
This notebook has been written for an audience already familiar with TIP. For users completely new to the platform, we recommend to follow the [TIP basic course](https://e-courses.epo.org/course/view.php?id=416) before reading this analysis.

## Disclaimer
This notebook is published as a showcase of the capabilities of TIP as a data processing environment. It is not intended to derive any statistical conclusion other than the presentation of the retrieved data. The EPO is not expressing any opinion extrapolated from the analysis.

## Introduction

This notebook is a continuation of the analysis on emerging technologies. For this reason, the work carried out on the data in the previous notebook must also be incorporated here to ensure the continuity of the analysis and to focus specifically on Europe.

It should be noted that this analysis is deliberately simplified, serving as a starting point to guide a more detailed exploration of Europe’s positioning within the global innovation landscape. This analysis was conducted in February 2025, and the results discussed reflect the data available at that time. The process can be repeated, but it is important to consider that results may vary over time.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS231_INPADOC_LEGAL_EVENT,
    TLS803_LEGAL_EVENT_CODE,
    TLS224_APPLN_CPC,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS225_DOCDB_FAM_CPC,
    TLS801_COUNTRY

)


patstat = PatstatClient(env="PROD")

db = patstat.orm()

In [2]:
from sqlalchemy import and_, case, func, select, distinct, or_
import requests
import zipfile
import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## CPC system

In [3]:
cpc_group_count_query = (
    db.query(
        TLS225_DOCDB_FAM_CPC.cpc_class_symbol, 
        func.count(TLS225_DOCDB_FAM_CPC.docdb_family_id).label("cpc_count")  
    )
    .join(
        TLS201_APPLN, TLS225_DOCDB_FAM_CPC.docdb_family_id == TLS201_APPLN.docdb_family_id  
    )
    .filter(
        TLS201_APPLN.earliest_publn_year >= 2013,
        TLS201_APPLN.earliest_publn_year != 9999
    )
    .group_by(TLS225_DOCDB_FAM_CPC.cpc_class_symbol)  
    .order_by(func.count(TLS225_DOCDB_FAM_CPC.docdb_family_id).desc())  
)


cpc_group_count_df = patstat.df(cpc_group_count_query)



cpc_group_count_df = cpc_group_count_df.head(110)



In [4]:
top_cpc_classes = cpc_group_count_df['cpc_class_symbol'].tolist()

top_cpc_classes[:50]  

['A61P  35/00',
 'Y02E  60/10',
 'A61K  45/06',
 'A61K2039/505',
 'A61P  43/00',
 'Y02P  70/50',
 'A61P  29/00',
 'G06N   3/08',
 'A23V2002/00',
 'Y02T  10/70',
 'Y02E  10/50',
 'Y02P  70/10',
 'H01M  10/0525',
 'Y02A  50/30',
 'A61P  25/00',
 'C07D 471/04',
 'G06N  20/00',
 'C07K2317/92',
 'Y02D  30/70',
 'G06N   3/045',
 'C07K2317/76',
 'Y02D  10/00',
 'C07D 487/04',
 'A61P  25/28',
 'A61K  38/00',
 'A61K   9/0019',
 'C07K2317/24',
 'H04N  19/176',
 'C07D 401/14',
 'H04W  72/23',
 'H01M  10/052',
 'A61P   3/10',
 'H04L   5/0053',
 'G06V  10/82',
 'A61P  11/00',
 'C07K2317/565',
 'A61P   9/00',
 'A61P  35/02',
 'Y02E  60/50',
 'Y02T  10/7072',
 'Y02T  10/12',
 'A61K  31/519',
 'A61P  31/04',
 'Y02E  10/72',
 'F21Y2115/10',
 'A61P  17/00',
 'H04N  19/70',
 'Y02P  10/20',
 'C07D 401/12',
 'H01M2220/20']

## IPFs for identifying **high-value inventions**

In [5]:
ipf_families_subquery = (
    db.query(TLS201_APPLN.docdb_family_id)
    .filter(
        # Families belonging to organisations
        TLS201_APPLN.docdb_family_id.in_(
            db.query(TLS201_APPLN.docdb_family_id)
            .join(TLS801_COUNTRY, TLS201_APPLN.appln_auth == TLS801_COUNTRY.ctry_code)
            .filter(TLS801_COUNTRY.organisation_flag == 'y')
            .distinct()
        )
        | 
        # Families with applications from 2+ distinct authorities
        TLS201_APPLN.docdb_family_id.in_(
            db.query(TLS201_APPLN.docdb_family_id)
            .filter(TLS201_APPLN.ipr_type != 'UM')  # Exclude utility models
            .group_by(TLS201_APPLN.docdb_family_id)
            .having(func.count(func.distinct(TLS201_APPLN.appln_auth)) >= 2)
        )
    )
    .distinct()
    .subquery()
)




ipf_families_query = (
    db.query(
        TLS201_APPLN.appln_id,               
        TLS201_APPLN.docdb_family_id,        
        TLS225_DOCDB_FAM_CPC.cpc_class_symbol,  
        TLS206_PERSON.person_ctry_code,
        TLS206_PERSON.person_id, 
        TLS206_PERSON.han_name, 
        TLS206_PERSON.psn_name, 
        TLS201_APPLN.earliest_publn_year,
        TLS206_PERSON.nuts,       
        TLS206_PERSON.nuts_level 
    )
    .join(
        TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id 
    )
    .join(
        TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id  
    )
    .join(
        TLS225_DOCDB_FAM_CPC, TLS201_APPLN.docdb_family_id == TLS225_DOCDB_FAM_CPC.docdb_family_id  
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(ipf_families_subquery),  
        TLS201_APPLN.earliest_publn_year >= 2013,   
        TLS201_APPLN.earliest_publn_year <= 2022,   
        TLS201_APPLN.earliest_publn_year != 9999,   
        TLS225_DOCDB_FAM_CPC.cpc_class_symbol.in_(top_cpc_classes),  
        TLS207_PERS_APPLN.applt_seq_nr != 0,       
        TLS206_PERSON.person_ctry_code.isnot(None),  
        TLS206_PERSON.person_ctry_code.in_([
            "US", "EP", "JP", "CN", "KR", "AL", "AT", "BE", "BG", "HR", "CY", 
            "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IS", "IE", "IT", 
            "LV", "LI", "LT", "LU", "MT", "MC", "NL", "MK", "NO", "PL", "PT", 
            "RO", "SM", "RS", "SK", "SI", "ES", "SE", "CH", "TR", "UK"
        ]), 
    )
    .order_by(TLS201_APPLN.docdb_family_id)  
)


ipf_families_df = patstat.df(ipf_families_query)


/tmp/ipykernel_1286/2644339836.py:50: SAWarning: Coercing Subquery object into a select() for use in IN(); please pass a select() construct explicitly
  TLS201_APPLN.docdb_family_id.in_(ipf_families_subquery),


In [6]:
ipf_families_df['cpc_class_prefix'] = ipf_families_df['cpc_class_symbol'].str[:4]

## Adding labels to the subclasses

In [7]:
prefixes = ipf_families_df['cpc_class_prefix'].unique()


In [8]:
from datetime import datetime

def get_current_CPCEdition():
    """
    Automatically determines the CPC edition based on today's date.

    Rule:
    - If the current month >= 8 (August) → use edition "YYYY08"
    - If the current month < 8 → use edition "YYYY01"
    """
    today = datetime.today()
    year = today.year

    if today.month >= 8:
        return f"{year}08"
    else:
        return f"{year}01"

def openCPCFiles(CPCEdition, prefixes):
    """
    Download and process all text files from the specified CPC ZIP folder.
    
    Args:
        CPCEdition (str): The CPC edition year and month.
        prefixes (list): A list of CPC class prefixes to filter the lines.
    
    Returns:
        pd.DataFrame: A DataFrame containing rows filtered by the given prefixes.
    """
    # URL of the ZIP file on the CPC website
    url = f"https://www.cooperativepatentclassification.org/sites/default/files/cpc/bulk/CPCTitleList{CPCEdition}.zip" 

    # Download ZIP file
    response = requests.get(url)
    if response.status_code == 200:
        # Open ZIP file from response content
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            # Initialize a list to hold data
            data = []
            # Iterate over all files in the ZIP
            for file_name in z.namelist():
                # Open and read each text file
                with z.open(file_name) as file:
                    # Read lines from the file
                    lines = file.readlines()
                    # Process all lines (no skipping)
                    for line in lines:
                        decoded_line = line.decode('utf-8').strip()
                        # Check if the line starts with any of the prefixes followed by a tab or '/'
                        if any(decoded_line.startswith(prefix + "\t") or decoded_line.startswith(prefix + "/") for prefix in prefixes):
                            data.append({'FileName': file_name, 'LineContent': decoded_line})

            # Convert the list to a DataFrame
            df = pd.DataFrame(data)
            return df
    else:
        print(f"Failed to download file: {response.status_code}")
        return None


cpc_edition = get_current_CPCEdition()
cpc_data = openCPCFiles(cpc_edition, prefixes)

if cpc_data is not None:
    print(cpc_data)

                      FileName  \
0   cpc-section-A_20260101.txt   
1   cpc-section-A_20260101.txt   
2   cpc-section-A_20260101.txt   
3   cpc-section-A_20260101.txt   
4   cpc-section-B_20260101.txt   
5   cpc-section-B_20260101.txt   
6   cpc-section-C_20260101.txt   
7   cpc-section-C_20260101.txt   
8   cpc-section-C_20260101.txt   
9   cpc-section-C_20260101.txt   
10  cpc-section-C_20260101.txt   
11  cpc-section-C_20260101.txt   
12  cpc-section-F_20260101.txt   
13  cpc-section-G_20260101.txt   
14  cpc-section-G_20260101.txt   
15  cpc-section-G_20260101.txt   
16  cpc-section-G_20260101.txt   
17  cpc-section-G_20260101.txt   
18  cpc-section-G_20260101.txt   
19  cpc-section-H_20260101.txt   
20  cpc-section-H_20260101.txt   
21  cpc-section-H_20260101.txt   
22  cpc-section-H_20260101.txt   
23  cpc-section-H_20260101.txt   
24  cpc-section-Y_20260101.txt   
25  cpc-section-Y_20260101.txt   
26  cpc-section-Y_20260101.txt   
27  cpc-section-Y_20260101.txt   
28  cpc-sectio

In [9]:
def processLineContent(df):
    df[['Prefix', 'Label']] = df['LineContent'].str.extract(r'([A-Z0-9/]+)\s+(.*)', expand=True)
    df['Prefix'] = df['Prefix'].apply(lambda x: x[:4])
    df['Label'] = df['Label'].str.replace(r'[^\x00-\x7F]+', '', regex=True)  

    return df
    
if cpc_data is not None:
    cpc_data = processLineContent(cpc_data)
    print(cpc_data)

                      FileName  \
0   cpc-section-A_20260101.txt   
1   cpc-section-A_20260101.txt   
2   cpc-section-A_20260101.txt   
3   cpc-section-A_20260101.txt   
4   cpc-section-B_20260101.txt   
5   cpc-section-B_20260101.txt   
6   cpc-section-C_20260101.txt   
7   cpc-section-C_20260101.txt   
8   cpc-section-C_20260101.txt   
9   cpc-section-C_20260101.txt   
10  cpc-section-C_20260101.txt   
11  cpc-section-C_20260101.txt   
12  cpc-section-F_20260101.txt   
13  cpc-section-G_20260101.txt   
14  cpc-section-G_20260101.txt   
15  cpc-section-G_20260101.txt   
16  cpc-section-G_20260101.txt   
17  cpc-section-G_20260101.txt   
18  cpc-section-G_20260101.txt   
19  cpc-section-H_20260101.txt   
20  cpc-section-H_20260101.txt   
21  cpc-section-H_20260101.txt   
22  cpc-section-H_20260101.txt   
23  cpc-section-H_20260101.txt   
24  cpc-section-Y_20260101.txt   
25  cpc-section-Y_20260101.txt   
26  cpc-section-Y_20260101.txt   
27  cpc-section-Y_20260101.txt   
28  cpc-sectio

In [10]:
cpc_data.rename(columns={'Prefix': 'cpc_class_prefix'}, inplace=True)

merged_data = pd.merge(ipf_families_df, cpc_data[['cpc_class_prefix', 'Label']], 
                       on='cpc_class_prefix', 
                       how='left')


## WIPO - IPC Concordance table to identify the fields

In [11]:
def clean_IPC(x):
    x = x.replace("/", "").replace("%", "").replace(" ", "")
    x = x[:4]
    return x

In [12]:
url = "https://www.wipo.int/documents/2948119/3215563/ipc_technology.xlsx"
techno_df = pd.read_excel(
    url,
    skiprows=6,
    usecols=['Field_number', 'Sector_en', 'Field_en', 'IPC_code'],
    converters={"IPC_code": clean_IPC},
)
techno_df.columns = ['Field Nb', 'Sector', 'Field', 'IPC']

In [13]:
techno_df = (techno_df.groupby(["Field Nb", "Sector", "Field"])["IPC"].agg(", ".join).reset_index())


In [14]:
ipc_to_field_mapping = {}
for _, row in techno_df.iterrows():
    for ipc_code in row['IPC'].split(", "):
        ipc_to_field_mapping[ipc_code.strip()] = row['Field']

def assign_field(cpc_prefix):
    if cpc_prefix in ipc_to_field_mapping:
        return ipc_to_field_mapping[cpc_prefix]
    if cpc_prefix.startswith("Y02"):
        return "Climate change mitigation"
    return "UNKNOWN" 

merged_data["Field"] = merged_data["cpc_class_prefix"].apply(assign_field)
merged_data = merged_data[merged_data["Field"] != "UNKNOWN"]


# Focus on Europe 

The rapid evolution of *emerging technologies* such as **computer technology**, **medical technology**, and **digital communication** has become a key driver of global innovation, particularly in Europe. These fields, characterised by their transformative impact on industries, economies, and societies, have witnessed significant growth in recent years. Europe plays a pivotal role in this innovation landscape, contributing substantially to the development and advancement of these technologies. The **European Patent Office (EPO)**, through its extensive patenting activities, provides valuable insights into these technological domains, offering a unique perspective on the contributions made by European countries to the global patenting ecosystem. This report delves into the patenting activity in Europe, specifically focusing on the inventions originating from the *European Patent Convention (EPC)* contracting states. It explores the contributions of various European countries to key sectors such as **computer technology**, **medical technology**, and **digital communication**, shedding light on the regional distribution of innovation in these fields.

By analysing the patenting trends within the *EPC contracting states*, this report highlights the role of individual countries in driving innovation across these emerging technologies. The data presented in the following sections offers a comprehensive view of the patent families associated with these fields, broken down by country, providing a clear understanding of the strengths and areas of excellence within Europe. Additionally, the report compares the European contributions to the broader **IP5 countries** (the United States, Japan, China, South Korea, and the European Patent Office), offering insights into Europe’s competitive position in these critical areas of technological development. Through this analysis, the report aims to offer policymakers, industry leaders, and researchers valuable information for strategic decision-making, investment planning, and collaboration opportunities within the rapidly evolving innovation landscape of Europe.


In [15]:
computer_technology_data = merged_data[merged_data['Field'] == 'Computer technology']
medical_technology_data = merged_data[merged_data['Field'] == 'Medical technology']
digital_communication_data = merged_data[merged_data['Field'] == 'Digital communication']

### Computer Technology

In [16]:
european_countries = [
    "AL", "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", 
    "GR", "HU", "IS", "IE", "IT", "LV", "LI", "LT", "LU", "MT", "MC", "NL", 
    "MK", "NO", "PL", "PT", "RO", "SM", "RS", "SK", "SI", "ES", "SE", "CH", 
    "TR", "UK"
]

total_unique_families = computer_technology_data['docdb_family_id'].nunique()

europe_unique_families = computer_technology_data[
    computer_technology_data['person_ctry_code'].isin(european_countries)
]['docdb_family_id'].nunique()

total_unique_families, europe_unique_families


(77865, 11402)

It could also be interesting to focus on **Europe** regarding certain *emerging fields*. Continuing with the interest in fields such as **computer technology**, **medical technology**, and **digital communication**, we will examine the European situation. Specifically, we will analyse the inventions originating from the *contracting states*, divided by country. In particular, inventions from the **IP5 countries** amount to 87,385, all of which are attributed to the contracting states.


In [17]:
european_data = computer_technology_data[
    computer_technology_data['person_ctry_code'].isin(european_countries)
]

unique_families_per_country = european_data.groupby('person_ctry_code')['docdb_family_id'].nunique().reset_index()

unique_families_per_country.columns = ['Country', 'Unique Families']

ranked_countries = unique_families_per_country.sort_values(by='Unique Families', ascending=False).reset_index(drop=True)

ranked_countries['Share of Total Unique Families in the IP5 Countries (%)'] = (
    ranked_countries['Unique Families'] / total_unique_families * 100
).round(1)


ranked_countries['Share of Europe Unique Families (%)'] = (
    ranked_countries['Unique Families'] / europe_unique_families * 100
).round(1)
ranked_countries

,Country,Unique Families,Share of Total Unique Families in the IP5 Countries (%),Share of Europe Unique Families (%)
0,DE,4287,5.5,37.6
1,FR,1718,2.2,15.1
2,NL,1317,1.7,11.6
3,CH,921,1.2,8.1
4,SE,832,1.1,7.3
5,FI,741,1.0,6.5
6,IE,528,0.7,4.6
7,IT,286,0.4,2.5
8,BE,228,0.3,2.0
9,ES,169,0.2,1.5


Innovation in **computer technology** within the **EPC (European Patent Convention)** countries is significantly influenced by **Germany**, which stands out as the leading contributor. In fact, **Germany** alone accounts for 33.3% of all unique patent families (IPFs) in Europe, indicating its dominant role in driving innovation within the region. On the global scale of the **IP5 countries**, which includes the major patenting nations (*China*, *Japan*, *South Korea*, *the United States*, and *the European Patent Office*), **Germany** contributes nearly 5% of the total unique families.

Following **Germany**, other key European countries in the field of computer technology innovation include **France** and the **Netherlands**, with shares of 14.1% and 9.9% respectively in Europe. The Nordic countries such as **Finland** and **Sweden** also make notable contributions, with shares of 7.0% and 6.8% in Europe.

Countries like **Switzerland** and **Ireland** also have significant roles, contributing 6.7% and 3.8% respectively to the total unique patent families in Europe. While other European nations like **Italy**, **Belgium**, and **Spain** have smaller shares, they still represent important hubs of innovation in the sector.


### Medical Technology

In [18]:
total_unique_families = medical_technology_data['docdb_family_id'].nunique()

europe_unique_families = medical_technology_data[
    medical_technology_data['person_ctry_code'].isin(european_countries)
]['docdb_family_id'].nunique()

total_unique_families, europe_unique_families

(10032, 2268)

**Medical technology** is another area where **Europe** plays a pivotal role in global innovation. The **IP5 countries** collectively account for 9,899 inventions in this sector, showcasing a strong presence in advancing medical technologies worldwide. Within the **EPC contracting states**, the total number of inventions in medical technology amounts to **2,254**, further underscoring Europe's significant contribution to this field.


In [19]:
european_data = medical_technology_data[
    medical_technology_data['person_ctry_code'].isin(european_countries)
]

unique_families_per_country = european_data.groupby('person_ctry_code')['docdb_family_id'].nunique().reset_index()

unique_families_per_country.columns = ['Country', 'Unique Families']




ranked_countries = unique_families_per_country.sort_values(by='Unique Families', ascending=False).reset_index(drop=True)

ranked_countries['Share of Total Unique Families in the 5 Countries (%)'] = (
    ranked_countries['Unique Families'] / total_unique_families * 100
).round(1)


ranked_countries['Share of Europe Unique Families (%)'] = (
    ranked_countries['Unique Families'] / europe_unique_families * 100
).round(1)
ranked_countries

,Country,Unique Families,Share of Total Unique Families in the 5 Countries (%),Share of Europe Unique Families (%)
0,NL,670,6.7,29.5
1,DE,544,5.4,24.0
2,CH,331,3.3,14.6
3,FR,253,2.5,11.2
4,IE,98,1.0,4.3
5,SE,93,0.9,4.1
6,FI,68,0.7,3.0
7,IT,62,0.6,2.7
8,DK,58,0.6,2.6
9,BE,54,0.5,2.4


Within the **EPC contracting states**, the **Netherlands** leads the way with **670 unique families**, representing **29.7%** of all European medical technology inventions. **Germany** follows closely, contributing **541 unique families**, or **24.0%** of Europe's total, underscoring its leadership in this innovative sector. Other notable contributors include **Switzerland** (**325 families, 14.4%**), **France** (**252 families, 11.2%**), and **Ireland** (**96 families, 4.3%**). Together, these countries form the backbone of Europe's medical technology innovation landscape, which is responsible for significant advancements in **diagnostics**, **medical devices**, and **healthcare systems**.


### Digital Communication

In [20]:
total_unique_families = digital_communication_data['docdb_family_id'].nunique()

europe_unique_families = digital_communication_data[
    digital_communication_data['person_ctry_code'].isin(european_countries)
]['docdb_family_id'].nunique()

total_unique_families, europe_unique_families


(94708, 14083)

In the field of **digital communication**, Europe plays a prominent role in global innovation, contributing significantly to the development of technologies that drive **connectivity** and **information exchange**. The **IP5 countries** collectively account for **90,972 unique families** in this sector, while the **EPC contracting states** contribute **13,446 unique families**, further demonstrating Europe's importance in advancing **digital communication technologies**.


In [21]:
european_data = digital_communication_data[
    digital_communication_data['person_ctry_code'].isin(european_countries)
]

unique_families_per_country = european_data.groupby('person_ctry_code')['docdb_family_id'].nunique().reset_index()

unique_families_per_country.columns = ['Country', 'Unique Families']

ranked_countries = unique_families_per_country.sort_values(by='Unique Families', ascending=False).reset_index(drop=True)

ranked_countries['Share of Total Unique Families in the 5 Countries (%)'] = (
    ranked_countries['Unique Families'] / total_unique_families * 100
).round(1)


ranked_countries['Share of Europe Unique Families (%)'] = (
    ranked_countries['Unique Families'] / europe_unique_families * 100
).round(1)
ranked_countries

,Country,Unique Families,Share of Total Unique Families in the 5 Countries (%),Share of Europe Unique Families (%)
0,SE,4826,5.1,34.3
1,DE,2825,3.0,20.1
2,FI,2401,2.5,17.0
3,FR,2098,2.2,14.9
4,NL,772,0.8,5.5
5,CH,490,0.5,3.5
6,IE,230,0.2,1.6
7,IT,174,0.2,1.2
8,DK,139,0.1,1.0
9,AT,113,0.1,0.8


Among the **EPC contracting states**, **Sweden** stands out with **4,575 unique families**, representing **34.0%** of Europe's total digital communication inventions. This is followed by **Germany**, with **2,747 unique families** (or **20.4%**), and **Finland**, contributing **2,269 unique families** (or **16.9%**). These countries, along with **France** and **the Netherlands**, make up the top contributors, each playing a vital role in driving technological advancements in **digital communication**, from **telecommunications networks** to **next-generation internet infrastructure**.

Smaller contributions come from countries such as **Switzerland**, **Ireland**, and **Italy**, but their impact remains significant. This distribution reflects the widespread innovation across Europe in the field, highlighting that digital communication is not dominated by any single country but is a collaborative effort across the region.


## Leveraging NUTS level


Examining the **origins of inventions** at a granular level provides valuable insights into **regional innovation patterns**, highlighting areas of technological activity and development. One effective approach to achieve this is by structuring the data according to **NUTS** (Nomenclature of Units for Territorial Statistics) codes, specifically **NUTS Level 3**. This level offers a detailed geographic breakdown, enabling a clearer understanding of where patents originate within specific regions. By mapping inventions to **NUTS Level 3**, it is possible to uncover localised clusters of innovation, identify key hubs of technological progress, and observe how certain regions contribute to broader trends in global innovation. This approach can reveal not only the intensity of invention in a region but also its technological focus and expertise.

For even greater precision, **NUTS Level 4** provides a finer level of granularity, offering an equivalent breakdown of regions at a more specific scale. These codes are available through the **OECD's REGPAT database**, further enhancing the ability to trace the origins of inventions to individual localities within a country. This detailed geographic perspective allows for a deeper understanding of how innovation is distributed across Europe, offering policymakers, businesses, and researchers a more refined tool to analyse **regional strengths** and potential areas for growth. By leveraging both **NUTS Level 3** and **Level 4** data, it becomes possible to not only identify the regions where inventions arise but also to explore the dynamics of **regional innovation networks** and their impact on the global technology landscape.


In [22]:
computer_technology_data_eu = computer_technology_data[computer_technology_data['nuts_level'].isin([3, 4])]
computer_technology_data_eu

,appln_id,docdb_family_id,cpc_class_symbol,person_ctry_code,person_id,han_name,psn_name,earliest_publn_year,nuts,nuts_level,cpc_class_prefix,Label,Field
10668,445750558,17440575,G06F 3/0482,IE,69888835,DATA SCAPE LTD,DATASCAPE,2016,IE052,4,G06F,ELECTRIC DIGITAL DATA PROCESSING (computer sys...,Computer technology
13220,487074289,20302367,G06F 3/0482,SE,53446965,GAMBRO LUNDIA AB,GAMBRO LUNDIA,2019,SE224,3,G06F,ELECTRIC DIGITAL DATA PROCESSING (computer sys...,Computer technology
95959,421285949,32842750,G06F 3/0482,FR,577561,THOMSON LICENSING,THOMSON LICENSING,2015,FR105,3,G06F,ELECTRIC DIGITAL DATA PROCESSING (computer sys...,Computer technology
95960,421285949,32842750,G06F 3/0482,FR,577561,THOMSON LICENSING,THOMSON LICENSING,2015,FR105,3,G06F,ELECTRIC DIGITAL DATA PROCESSING (computer sys...,Computer technology
142262,445446779,34621554,G06F 3/0482,FR,10231,THOMSON LICENSING,THOMSON LICENSING,2016,FR105,3,G06F,ELECTRIC DIGITAL DATA PROCESSING (computer sys...,Computer technology
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12102804,493585468,89992283,G06F 30/20,HU,72229590,ECONOMETRIX KFT,ECONOMETRIX,2019,HU110,4,G06F,ELECTRIC DIGITAL DATA PROCESSING (computer sys...,Computer technology
12103359,504765547,89992519,G06N 3/045,HU,74461637,SOLECALL KFT,SOLECALL,2020,HU110,4,G06N,COMPUTING ARRANGEMENTS BASED ON SPECIFIC COMPU...,Computer technology
12103363,504765547,89992519,G06N 3/08,HU,74461637,SOLECALL KFT,SOLECALL,2020,HU110,4,G06N,COMPUTING ARRANGEMENTS BASED ON SPECIFIC COMPU...,Computer technology
12103365,504765547,89992519,G06N 3/084,HU,74461637,SOLECALL KFT,SOLECALL,2020,HU110,4,G06N,COMPUTING ARRANGEMENTS BASED ON SPECIFIC COMPU...,Computer technology


In [23]:
medical_technology_data_eu = medical_technology_data[medical_technology_data['nuts_level'].isin([3, 4])]
medical_technology_data_eu

,appln_id,docdb_family_id,cpc_class_symbol,person_ctry_code,person_id,han_name,psn_name,earliest_publn_year,nuts,nuts_level,cpc_class_prefix,Label,Field
183609,527990774,35431236,G16H 50/20,DK,74450258,SPIROFRIENDS TECH APS,SPIROFRIENDS TECHNOLOGY,2020,DK012,4,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
335431,457123240,38171510,G16H 50/20,DK,2784,NOVO NORDISK AS,NOVO NORDISK,2016,DK012,3,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
335432,457123240,38171510,G16H 50/20,DK,2784,NOVO NORDISK AS,NOVO NORDISK,2016,DK012,3,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
394936,517124156,38802240,G16H 50/20,IE,49594338,RESMED SENSOR TECH LTD,RESMED SENSOR TECHNOLOGIES,2020,IE021,3,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
394941,517124156,38802240,G16H 50/20,IE,49594338,RESMED SENSOR TECH LTD,RESMED SENSOR TECHNOLOGIES,2020,IE021,3,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12074890,547217595,84179634,G16H 50/20,CH,74421362,EPYMETRICS AG,EPYMETRICS,2022,CH040,4,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
12074891,547217595,84179634,G16H 50/20,CH,74421362,EPYMETRICS AG,EPYMETRICS,2022,CH040,4,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
12097759,528123676,89719866,G16H 50/20,HU,47030691,SZEGEDI TUDOMANYEGYETEM,SZEGEDI TUDOMÁNYEGYETEM (SZTE),2021,HU333,3,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology
12103380,499402678,89992568,G16H 50/20,HU,69903699,KOZMANN GYORGY ZOLTAN,KOZMANN GYOERGY ZOLTAN,2020,HU110,4,G16H,"HEALTHCARE INFORMATICS, i.e. INFORMATION AND C...",Medical technology


In [24]:
digital_communication_data_eu = digital_communication_data[digital_communication_data['nuts_level'].isin([3, 4])]
digital_communication_data_eu

,appln_id,docdb_family_id,cpc_class_symbol,person_ctry_code,person_id,han_name,psn_name,earliest_publn_year,nuts,nuts_level,cpc_class_prefix,Label,Field
2230,420216051,7900181,H04W 72/23,DE,21880,IPCOM GMBH & CO KG,IPCOM & COMPANY,2014,DE21H,3,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
2231,378467439,7900181,H04W 72/23,DE,21880,IPCOM GMBH & CO KG,IPCOM & COMPANY,2013,DE21H,3,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
2234,420216051,7900181,H04W 72/23,DE,21880,IPCOM GMBH & CO KG,IPCOM & COMPANY,2014,DE21H,3,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
2236,378467433,7900181,H04W 72/23,DE,21880,IPCOM GMBH & CO KG,IPCOM & COMPANY,2013,DE21H,3,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
2237,378467433,7900181,H04W 72/23,DE,21880,IPCOM GMBH & CO KG,IPCOM & COMPANY,2013,DE21H,3,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12105998,443258037,91081887,H04W 72/23,FR,47187361,"Stojanovski, Alexandre Saso","STOJANOVSKI, ALEXANDRE SASO",2015,FR101,4,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
12108294,448423745,94380282,H04W 72/23,DE,5217608,"Choi, Hyung-Nam","CHOI, HYUNG-NAM",2016,DE600,4,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication
12108304,448423745,94380282,H04L 5/0053,DE,5217608,"Choi, Hyung-Nam","CHOI, HYUNG-NAM",2016,DE600,4,H04L,"TRANSMISSION OF DIGITAL INFORMATION, e.g. TELE...",Digital communication
12108307,448423745,94380282,H04W 84/12,DE,5217608,"Choi, Hyung-Nam","CHOI, HYUNG-NAM",2016,DE600,4,H04W,WIRELESS COMMUNICATION NETWORKS (broadcast com...,Digital communication


## Geomapping of the inventions for three different emerging fields

To **localise the inventions**, the first step involves aggregating the innovations by **NUTS level**. This will provide a comprehensive overview of the **regional distribution** of patents, allowing us to identify areas with the highest concentrations of technological activity. Initially, this analysis will be conducted **without considering a temporal division**, enabling us to observe the overall distribution of inventions across regions at a specific point in time. The data will then be **mapped** to visually represent these **geographic patterns**, offering a clear depiction of where inventions are concentrated across various territories.

Following this initial mapping, the next step will be to explore the **temporal evolution** of these inventions. By examining how the **distribution of innovations** has changed over the years, we can gain deeper insights into the dynamics of **regional technological development**. This **temporal analysis** will highlight trends, such as the emergence of new **innovation hubs** or shifts in the focus of invention, allowing for a more nuanced understanding of how **regional innovation evolves** over time.


### Computer Technology

In [25]:

invention_count_per_nuts = (
    computer_technology_data_eu.groupby('nuts')['docdb_family_id'].nunique().reset_index()
)

invention_count_per_nuts.rename(columns={'docdb_family_id': 'invention_count'}, inplace=True)

invention_count_per_nuts


,nuts,invention_count
0,AT121,2
1,AT123,2
2,AT124,3
3,AT126,4
4,AT130,31
...,...,...
465,TR100,8
466,TR331,12
467,TR332,1
468,TR411,1


#### Adding time 

In [26]:
invention_count_per_nuts_year = (
    computer_technology_data_eu.groupby(['nuts', 'earliest_publn_year'])['docdb_family_id']
    .nunique()
    .reset_index()
)

invention_count_per_nuts_year.rename(columns={'docdb_family_id': 'invention_count'}, inplace=True)

invention_count_per_nuts_year


,nuts,earliest_publn_year,invention_count
0,AT121,2016,1
1,AT121,2021,2
2,AT123,2018,1
3,AT123,2021,1
4,AT124,2019,2
...,...,...,...
1422,TR510,2014,1
1423,TR510,2017,1
1424,TR510,2019,1
1425,TR510,2021,1


In [27]:
import os

import requests

GEOFILE_NAME = "NUTS_RG_20M_2024_3035.geojson"
FULL_GEOFILE_PATH = f"{os.environ.get("HOME")}/{GEOFILE_NAME}"

url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.json"

def fetch_nuts_metadata():
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()  
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None


metadata = fetch_nuts_metadata()

if metadata:
    
    file_path = metadata.get('geojson', {}).get(GEOFILE_NAME)
    
    if file_path:
        
        download_url = f"https://gisco-services.ec.europa.eu/distribution/v2/nuts/{file_path}"
        
       
        response = requests.get(download_url)
        if response.status_code == 200:
            # Save the downloaded GeoJSON file to local disk
            with open(FULL_GEOFILE_PATH, "wb") as file:
                file.write(response.content)
            print("GeoJSON file downloaded successfully!")
        else:
            print(f"Failed to download GeoJSON file. Status code: {response.status_code}")
    else:
        print("Level 3 GeoJSON file not found in the metadata.")
else:
    print("Failed to fetch metadata.")

GeoJSON file downloaded successfully!


In [28]:
!pip install geopandas

import geopandas as gpd


gdf = gpd.read_file(FULL_GEOFILE_PATH)


gdf= gdf[gdf['LEVL_CODE'] == 3]


gdf = gdf[['NUTS_ID', 'CNTR_CODE', 'NAME_LATN', 'geometry']]

gdf

,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
0,AL011,AL,Dibër,"POLYGON ((5169746.713 2142724.535, 5198279.451..."
1,AL012,AL,Durrës,"POLYGON ((5118781.291 2103409.375, 5141704.829..."
2,AL013,AL,Kukës,"POLYGON ((5200419.912 2147880.017, 5198279.451..."
3,AL014,AL,Lezhë,"POLYGON ((5112296.856 2131907.92, 5108760.398 ..."
4,AL015,AL,Shkodër,"POLYGON ((5100056.794 2130793.954, 5098686.356..."
...,...,...,...,...
1793,RO224,RO,Galaţi,"POLYGON ((5719061.241 2647285.38, 5681166.966 ..."
1794,RO225,RO,Tulcea,"POLYGON ((5811317.429 2587520.168, 5774466.401..."
1795,RO226,RO,Vrancea,"POLYGON ((5674090.202 2635195.873, 5648732.311..."
1796,RO311,RO,Argeş,"POLYGON ((5456421.478 2525143.999, 5457788.06 ..."


In [29]:
merged_df = invention_count_per_nuts.merge(gdf, left_on='nuts', right_on='NUTS_ID', how='left')
merged_df = merged_df.sort_values(by='invention_count', ascending=False)
merged_df

,nuts,invention_count,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
89,DE212,650,DE212,DE,"München, Kreisfreie Stadt","POLYGON ((4426190.467 2780289.975, 4425325.764..."
415,NL414,567,NL414,NL,Zuidoost-Noord-Brabant,"POLYGON ((3980655.573 3157882.987, 3993416.951..."
254,FI1B1,442,FI1B1,FI,Helsinki-Uusimaa,"MULTIPOLYGON (((5158203.952 4278166.261, 51678..."
263,FR101,406,FR101,FR,Paris,"POLYGON ((3765368.749 2888183.414, 3758842.685..."
267,FR105,309,FR105,FR,Hauts-de-Seine,"POLYGON ((3757379.981 2900396.984, 3758912.17 ..."
...,...,...,...,...,...,...
65,DE11C,1,DE11C,DE,Heidenheim,"POLYGON ((4338058.152 2821924.278, 4316875.997..."
67,DE121,1,DE121,DE,"Baden-Baden, Stadtkreis","POLYGON ((4183478.603 2853567.776, 4194251.385..."
70,DE124,1,DE124,DE,Rastatt,"POLYGON ((4191568.421 2874149.063, 4193703.111..."
441,PT16E,1,NaN,NaN,NaN,None


The data on **computer technology** innovation showcases notable regional contributions within Europe. **München, Kreisfreie Stadt (DE212)** in Germany leads with **651 inventions**, underscoring its status as a prominent centre for technological advancements. **Zuidoost-Noord-Brabant (NL414)** in the Netherlands follows closely with **590 inventions**, highlighting the region's significant role in fostering innovation. 

Other key regions include **Helsinki-Uusimaa (FI1B1)** in Finland with **558 inventions**, further demonstrating the strength of Nordic countries in this domain. Additionally, **Paris (FR101)** in France and **Pfaffenhofen a. d. Ilm (DE21J)** in Germany contribute **443** and **393 inventions**, respectively, reflecting the diversity and widespread nature of innovation across Europe in the field of computer technology.


In [30]:
merged_df_year = invention_count_per_nuts_year.merge(gdf, left_on='nuts', right_on='NUTS_ID', how='left')  
merged_df_year

,nuts,earliest_publn_year,invention_count,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
0,AT121,2016,1,AT121,AT,Mostviertel-Eisenwurzen,"POLYGON ((4676275.622 2748475.808, 4675514.844..."
1,AT121,2021,2,AT121,AT,Mostviertel-Eisenwurzen,"POLYGON ((4676275.622 2748475.808, 4675514.844..."
2,AT123,2018,1,AT123,AT,Sankt Pölten,"POLYGON ((4729365.295 2815976.362, 4745246.483..."
3,AT123,2021,1,AT123,AT,Sankt Pölten,"POLYGON ((4729365.295 2815976.362, 4745246.483..."
4,AT124,2019,2,AT124,AT,Waldviertel,"POLYGON ((4688301.641 2819349.199, 4680120.542..."
...,...,...,...,...,...,...,...
1422,TR510,2014,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."
1423,TR510,2017,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."
1424,TR510,2019,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."
1425,TR510,2021,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."


In [31]:
merged_gdf = gpd.GeoDataFrame(merged_df, geometry='geometry')

# Reproject to EPSG:4326 (WGS84) - geographic coordinates (latitude, longitude)
merged_gdf = merged_gdf.to_crs(epsg=4326)

# Calculate centroids from geometry
merged_gdf['centroid'] = merged_gdf['geometry'].centroid

# Extract latitude and longitude from the centroids
merged_gdf['latitude'] = merged_gdf['centroid'].y
merged_gdf['longitude'] = merged_gdf['centroid'].x

# Ensure no NaN values are present
merged_gdf = merged_gdf.dropna(subset=['latitude', 'longitude', 'invention_count'])

/tmp/ipykernel_1286/3627937061.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  merged_gdf['centroid'] = merged_gdf['geometry'].centroid


In [32]:
merged_gdf_year = gpd.GeoDataFrame(merged_df_year, geometry='geometry')


# Reproject to EPSG:4326 (WGS84) - geographic coordinates (latitude, longitude)
merged_gdf_year = merged_gdf_year.to_crs(epsg=4326)

# Calculate centroids from geometry
merged_gdf_year['centroid'] = merged_gdf_year['geometry'].centroid

# Extract latitude and longitude from the centroids
merged_gdf_year['latitude'] = merged_gdf_year['centroid'].y
merged_gdf_year['longitude'] = merged_gdf_year['centroid'].x

# Ensure no NaN values are present
merged_gdf_year = merged_gdf_year.dropna(subset=['latitude', 'longitude', 'invention_count'])

/tmp/ipykernel_1286/1170087037.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  merged_gdf_year['centroid'] = merged_gdf_year['geometry'].centroid


In [33]:
import plotly.express as px

# Plot the centroids (using latitude and longitude)
fig = px.scatter_mapbox(
    merged_gdf,
    lat="latitude",  # Latitude of centroids
    lon="longitude",  # Longitude of centroids
    color="invention_count",  # Color based on application count
    hover_name="NAME_LATN",  # Hover information for each point
    size="invention_count",  # Size of marker based on application count
    color_continuous_scale="Viridis",  # Color scale for better visualisation
    mapbox_style="carto-positron",  # Map style
    center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    zoom=6,  # Zoom level
    title="Invention Counts by Region (Centroids)",  # Add title
    height=600,  # Height of the plot (you can adjust as needed)
    labels={"invention_count": "Number of Inventions"} 
)
# Update hover information to remove latitude and longitude and add a custom note
fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"  # Display the region name (NAME_LATN)
        "Applications: %{marker.size}<br>"  # Display the application count
        "(from 2000 to 2023)"  # Custom annotation
    )
)
# Make the map interactive: zoom, pan, hover
fig.update_layout(
    mapbox=dict(
        zoom=6,  # Initial zoom level
        style="carto-positron",  # Map style
        center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    ),
    title="Number of Inventions across Europe",
)

fig.show()


/tmp/ipykernel_1286/3384590069.py:4: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [34]:
import plotly.express as px
import pandas as pd

# Ensure 'invention_count' is numeric
merged_gdf_year['invention_count'] = pd.to_numeric(merged_gdf_year['invention_count'], errors='coerce')

# Drop any rows with NaN values in 'invention_count'
merged_gdf_year = merged_gdf_year.dropna(subset=['invention_count'])

# Ensure 'earliest_publn_year' is numeric (if it's not already)
merged_gdf_year['earliest_publn_year'] = pd.to_numeric(merged_gdf_year['earliest_publn_year'], errors='coerce')

# Sort by 'earliest_publn_year' to ensure chronological order
merged_gdf_year = merged_gdf_year.sort_values(by='earliest_publn_year')

# Plot the centroids (using latitude and longitude) and animate by 'earliest_publn_year'
fig = px.scatter_mapbox(
    merged_gdf_year,
    lat="latitude",  # Latitude of centroids
    lon="longitude",  # Longitude of centroids
    color="invention_count",  # Color based on application count
    size="invention_count",  # Size of marker based on application count
    color_continuous_scale="Viridis",  # Color scale for better visualisation
    mapbox_style="carto-positron",  # Map style
    animation_frame="earliest_publn_year",  # Animate by 'earliest_publn_year'
    animation_group="NUTS_ID",  # Ensure the same region stays together in each frame
    center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    zoom=6,  # Zoom level
    title="Invention Counts by Region (Centroids) Over Time",  # Add title
    height=600,  # Height of the plot (you can adjust as needed)
    labels={"invention_count": "Number of Inventions"}, 
    hover_name="NAME_LATN",  # Set the place name as the title at the top of hover
    hover_data={
        "latitude": False,  # Exclude latitude
        "longitude": False,  # Exclude longitude
        "invention_count": True,  # Include invention count
        "earliest_publn_year": True  # Include year
    }
)

# Make the map interactive: zoom, pan, hover
fig.update_layout(
    mapbox=dict(
        zoom=6,  # Initial zoom level
        style="carto-positron",  # Map style
        center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    ),
    title="Invention Counts by Region Over Time",
)

# Display the figure
fig.show()


/tmp/ipykernel_1286/57790570.py:17: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



## Medical Technology

In [35]:
invention_count_per_nuts = (
    medical_technology_data_eu.groupby('nuts')['docdb_family_id'].nunique().reset_index()
)

invention_count_per_nuts.rename(columns={'docdb_family_id': 'invention_count'}, inplace=True)

invention_count_per_nuts


,nuts,invention_count
0,AT126,1
1,AT130,6
2,AT211,1
3,AT221,1
4,AT312,1
...,...,...
232,SE125,2
233,SE224,10
234,SE232,10
235,TR100,2


In [36]:
invention_count_per_nuts_year = (
    medical_technology_data_eu.groupby(['nuts', 'earliest_publn_year'])['docdb_family_id']
    .nunique()
    .reset_index()
)

invention_count_per_nuts_year.rename(columns={'docdb_family_id': 'invention_count'}, inplace=True)

invention_count_per_nuts_year

,nuts,earliest_publn_year,invention_count
0,AT126,2021,1
1,AT130,2018,1
2,AT130,2019,2
3,AT130,2020,1
4,AT130,2021,3
...,...,...,...
589,SE232,2021,2
590,SE232,2022,2
591,TR100,2015,1
592,TR100,2021,1


In [37]:
merged_df = invention_count_per_nuts.merge(gdf, left_on='nuts', right_on='NUTS_ID', how='left')  
merged_df = merged_df.sort_values(by='invention_count', ascending=False)
merged_df

,nuts,invention_count,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
210,NL414,477,NL414,NL,Zuidoost-Noord-Brabant,"POLYGON ((3980655.573 3157882.987, 3993416.951..."
25,CH031,104,CH031,CH,Basel-Stadt,"POLYGON ((4139577.437 2722539.907, 4142867.509..."
121,FR101,94,FR101,FR,Paris,"POLYGON ((3765368.749 2888183.414, 3758842.685..."
58,DE252,93,DE252,DE,"Erlangen, Kreisfreie Stadt","POLYGON ((4388528.523 2936588.786, 4388373.983..."
45,DE126,49,DE126,DE,"Mannheim, Stadtkreis","POLYGON ((4206911.188 2942403.651, 4218269.222..."
...,...,...,...,...,...,...
224,PT150,1,PT150,PT,Algarve,"POLYGON ((2663470.619 1801233.066, 2690228.246..."
221,PL622,1,PL622,PL,Olsztyński,"POLYGON ((5069607.527 3528478.29, 5073723.389 ..."
222,PT112,1,PT112,PT,Cávado,"POLYGON ((2765263.289 2251928.613, 2786983.604..."
225,PT170,1,NaN,NaN,NaN,None


The data reveals a strong regional focus on **medical technology** innovation across Europe. **Zuidoost-Noord-Brabant (NL414)** in the Netherlands emerges as a leader with **476 inventions**, reflecting its critical role as a hub for medical technology advancements. **Erlangen, Kreisfreie Stadt (DE252)** in Germany follows with **104 inventions**, highlighting its contribution to this sector. **Basel-Stadt (CH031)** in Switzerland, known


In [38]:
merged_df_year = invention_count_per_nuts_year.merge(gdf, left_on='nuts', right_on='NUTS_ID', how='left')  
merged_df_year

,nuts,earliest_publn_year,invention_count,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
0,AT126,2021,1,AT126,AT,Wiener Umland/Nordteil,"POLYGON ((4748191.422 2835851.812, 4774680.49 ..."
1,AT130,2018,1,AT130,AT,Wien,"POLYGON ((4780587.347 2803485.343, 4784559.684..."
2,AT130,2019,2,AT130,AT,Wien,"POLYGON ((4780587.347 2803485.343, 4784559.684..."
3,AT130,2020,1,AT130,AT,Wien,"POLYGON ((4780587.347 2803485.343, 4784559.684..."
4,AT130,2021,3,AT130,AT,Wien,"POLYGON ((4780587.347 2803485.343, 4784559.684..."
...,...,...,...,...,...,...,...
589,SE232,2021,2,SE232,SE,Västra Götalands län,"POLYGON ((4405148.086 3988389.248, 4406696.655..."
590,SE232,2022,2,SE232,SE,Västra Götalands län,"POLYGON ((4405148.086 3988389.248, 4406696.655..."
591,TR100,2015,1,TR100,TR,İstanbul,"MULTIPOLYGON (((5978485.288 2224822.983, 59651..."
592,TR100,2021,1,TR100,TR,İstanbul,"MULTIPOLYGON (((5978485.288 2224822.983, 59651..."


In [39]:
merged_gdf = gpd.GeoDataFrame(merged_df, geometry='geometry')


# Reproject to EPSG:4326 (WGS84) - geographic coordinates (latitude, longitude)
merged_gdf = merged_gdf.to_crs(epsg=4326)

# Calculate centroids from geometry
merged_gdf['centroid'] = merged_gdf['geometry'].centroid

# Extract latitude and longitude from the centroids
merged_gdf['latitude'] = merged_gdf['centroid'].y
merged_gdf['longitude'] = merged_gdf['centroid'].x

# Ensure no NaN values are present
merged_gdf = merged_gdf.dropna(subset=['latitude', 'longitude', 'invention_count'])

/tmp/ipykernel_1286/2727612077.py:8: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [40]:
merged_df_year = gpd.GeoDataFrame(merged_df_year, geometry='geometry')


# Reproject to EPSG:4326 (WGS84) - geographic coordinates (latitude, longitude)
merged_df_year = merged_df_year.to_crs(epsg=4326)

# Calculate centroids from geometry
merged_df_year['centroid'] = merged_df_year['geometry'].centroid

# Extract latitude and longitude from the centroids
merged_df_year['latitude'] = merged_df_year['centroid'].y
merged_df_year['longitude'] = merged_df_year['centroid'].x

# Ensure no NaN values are present
merged_df_year = merged_df_year.dropna(subset=['latitude', 'longitude', 'invention_count'])

/tmp/ipykernel_1286/1401268086.py:8: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [41]:
# Plot the centroids (using latitude and longitude)
fig = px.scatter_mapbox(
    merged_gdf,
    lat="latitude",  # Latitude of centroids
    lon="longitude",  # Longitude of centroids
    color="invention_count",  # Color based on application count
    hover_name="NAME_LATN",  # Hover information for each point
    size="invention_count",  # Size of marker based on application count
    color_continuous_scale="Viridis",  # Color scale for better visualisation
    mapbox_style="carto-positron",  # Map style
    center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    zoom=6,  # Zoom level
    title="Invention Counts by Region (Centroids)",  # Add title
    height=600,  # Height of the plot (you can adjust as needed)
    labels={"invention_count": "Number of Inventions"} 
)
# Update hover information to remove latitude and longitude and add a custom note
fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"  # Display the region name (NAME_LATN)
        "Applications: %{marker.size}<br>"  # Display the application count
        "(from 2000 to 2023)"  # Custom annotation
    )
)
# Make the map interactive: zoom, pan, hover
fig.update_layout(
    mapbox=dict(
        zoom=6,  # Initial zoom level
        style="carto-positron",  # Map style
        center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    ),
    title="Number of Inventions across Europe",
)

fig.show()

/tmp/ipykernel_1286/1542089585.py:2: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [42]:
# Ensure 'invention_count' is numeric
merged_df_year['invention_count'] = pd.to_numeric(merged_df_year['invention_count'], errors='coerce')

# Drop any rows with NaN values in 'invention_count'
merged_df_year = merged_df_year.dropna(subset=['invention_count'])

# Ensure 'earliest_publn_year' is numeric (if it's not already)
merged_df_year['earliest_publn_year'] = pd.to_numeric(merged_df_year['earliest_publn_year'], errors='coerce')

# Sort by 'earliest_publn_year' to ensure chronological order
merged_df_year = merged_df_year.sort_values(by='earliest_publn_year')

# Plot the centroids (using latitude and longitude) and animate by 'earliest_publn_year'
fig = px.scatter_mapbox(
    merged_df_year,
    lat="latitude",  # Latitude of centroids
    lon="longitude",  # Longitude of centroids
    color="invention_count",  # Color based on application count
    size="invention_count",  # Size of marker based on application count
    color_continuous_scale="Viridis",  # Color scale for better visualisation
    mapbox_style="carto-positron",  # Map style
    animation_frame="earliest_publn_year",  # Animate by 'earliest_publn_year'
    animation_group="NUTS_ID",  # Ensure the same region stays together in each frame
    center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    zoom=6,  # Zoom level
    title="Invention Counts by Region (Centroids) Over Time",  # Add title
    height=600,  # Height of the plot (you can adjust as needed)
    labels={"invention_count": "Number of Inventions"}, 
    hover_name="NAME_LATN",  # Set the place name as the title at the top of hover
    hover_data={
        "latitude": False,  # Exclude latitude
        "longitude": False,  # Exclude longitude
        "invention_count": True,  # Include invention count
        "earliest_publn_year": True  # Include year
    }
)

# Make the map interactive: zoom, pan, hover
fig.update_layout(
    mapbox=dict(
        zoom=6,  # Initial zoom level
        style="carto-positron",  # Map style
        center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    ),
    title="Invention Counts by Region Over Time",
)

# Display the figure
fig.show()

/tmp/ipykernel_1286/3781347528.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



## Digital Communication

In [43]:
invention_count_per_nuts = (
    digital_communication_data_eu.groupby('nuts')['docdb_family_id'].nunique().reset_index()
)

invention_count_per_nuts.rename(columns={'docdb_family_id': 'invention_count'}, inplace=True)

invention_count_per_nuts

,nuts,invention_count
0,AT121,2
1,AT127,1
2,AT130,34
3,AT221,9
4,AT225,6
...,...,...
381,SK021,1
382,TR100,20
383,TR331,9
384,TR510,3


In [44]:
invention_count_per_nuts_year = (
    digital_communication_data_eu.groupby(['nuts', 'earliest_publn_year'])['docdb_family_id']
    .nunique()
    .reset_index()
)

invention_count_per_nuts_year.rename(columns={'docdb_family_id': 'invention_count'}, inplace=True)

invention_count_per_nuts_year

,nuts,earliest_publn_year,invention_count
0,AT121,2020,1
1,AT121,2022,1
2,AT127,2016,1
3,AT130,2013,4
4,AT130,2014,3
...,...,...,...
1236,TR331,2022,4
1237,TR510,2015,1
1238,TR510,2018,1
1239,TR510,2019,1


In [45]:
merged_df = invention_count_per_nuts.merge(gdf, left_on='nuts', right_on='NUTS_ID', how='left')  
merged_df = merged_df.sort_values(by='invention_count', ascending=False)
merged_df

,nuts,invention_count,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
368,SE110,3466,SE110,SE,Stockholms län,"MULTIPOLYGON (((4863571.028 4075519.853, 48616..."
210,FI1B1,1463,FI1B1,FI,Helsinki-Uusimaa,"MULTIPOLYGON (((5158203.952 4278166.261, 51678..."
220,FR105,694,FR105,FR,Hauts-de-Seine,"POLYGON ((3757379.981 2900396.984, 3758912.17 ..."
81,DE212,597,DE212,DE,"München, Kreisfreie Stadt","POLYGON ((4426190.467 2780289.975, 4425325.764..."
339,NL414,417,NL414,NL,Zuidoost-Noord-Brabant,"POLYGON ((3980655.573 3157882.987, 3993416.951..."
...,...,...,...,...,...,...
346,NO020,1,NO020,NO,Innlandet,"POLYGON ((4438332.47 4360687.11, 4440904.738 4..."
347,NO031,1,NaN,NaN,NaN,None
348,NO032,1,NaN,NaN,NaN,None
352,NO071,1,NO071,NO,Nordland/Nordlánnda,"MULTIPOLYGON (((4655620.252 5065448.583, 46603..."


The most active **NUTS regions** for **digital communication** include **Stockholms län (SE110)** in Sweden, which leads with **3,286 unique families** of inventions. **Helsinki-Uusimaa (FI1B1)** in Finland follows closely with **1,386 unique families**. **Hauts-de-Seine (FR105)** in France also stands out with **656 unique families** of inventions, highlighting the **prominence** of these regions in driving **innovation in digital communication**.


In [46]:
merged_df_year = invention_count_per_nuts_year.merge(gdf, left_on='nuts', right_on='NUTS_ID', how='left')  
merged_df_year

,nuts,earliest_publn_year,invention_count,NUTS_ID,CNTR_CODE,NAME_LATN,geometry
0,AT121,2020,1,AT121,AT,Mostviertel-Eisenwurzen,"POLYGON ((4676275.622 2748475.808, 4675514.844..."
1,AT121,2022,1,AT121,AT,Mostviertel-Eisenwurzen,"POLYGON ((4676275.622 2748475.808, 4675514.844..."
2,AT127,2016,1,AT127,AT,Wiener Umland/Südteil,"POLYGON ((4761472.615 2792467.94, 4766617.881 ..."
3,AT130,2013,4,AT130,AT,Wien,"POLYGON ((4780587.347 2803485.343, 4784559.684..."
4,AT130,2014,3,AT130,AT,Wien,"POLYGON ((4780587.347 2803485.343, 4784559.684..."
...,...,...,...,...,...,...,...
1236,TR331,2022,4,TR331,TR,Manisa,"POLYGON ((5922804.837 1984028.211, 5926327.551..."
1237,TR510,2015,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."
1238,TR510,2018,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."
1239,TR510,2019,1,TR510,TR,Ankara,"POLYGON ((6191602.422 2054885.177, 6197190.46 ..."


In [47]:
merged_gdf = gpd.GeoDataFrame(merged_df, geometry='geometry')


# Reproject to EPSG:4326 (WGS84) - geographic coordinates (latitude, longitude)
merged_gdf = merged_gdf.to_crs(epsg=4326)

# Calculate centroids from geometry
merged_gdf['centroid'] = merged_gdf['geometry'].centroid

# Extract latitude and longitude from the centroids
merged_gdf['latitude'] = merged_gdf['centroid'].y
merged_gdf['longitude'] = merged_gdf['centroid'].x

# Ensure no NaN values are present
merged_gdf = merged_gdf.dropna(subset=['latitude', 'longitude', 'invention_count'])

/tmp/ipykernel_1286/2727612077.py:8: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [48]:
merged_df_year = gpd.GeoDataFrame(merged_df_year, geometry='geometry')


# Reproject to EPSG:4326 (WGS84) - geographic coordinates (latitude, longitude)
merged_df_year = merged_df_year.to_crs(epsg=4326)

# Calculate centroids from geometry
merged_df_year['centroid'] = merged_df_year['geometry'].centroid

# Extract latitude and longitude from the centroids
merged_df_year['latitude'] = merged_df_year['centroid'].y
merged_df_year['longitude'] = merged_df_year['centroid'].x

# Ensure no NaN values are present
merged_df_year = merged_df_year.dropna(subset=['latitude', 'longitude', 'invention_count'])

/tmp/ipykernel_1286/1401268086.py:8: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [49]:
# Plot the centroids (using latitude and longitude)
fig = px.scatter_mapbox(
    merged_gdf,
    lat="latitude",  # Latitude of centroids
    lon="longitude",  # Longitude of centroids
    color="invention_count",  # Color based on application count
    hover_name="NAME_LATN",  # Hover information for each point
    size="invention_count",  # Size of marker based on application count
    color_continuous_scale="Viridis",  # Color scale for better visualisation
    mapbox_style="carto-positron",  # Map style
    center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    zoom=6,  # Zoom level
    title="Invention Counts by Region (Centroids)",  # Add title
    height=600,  # Height of the plot (you can adjust as needed)
    labels={"invention_count": "Number of Inventions"} 
)
# Update hover information to remove latitude and longitude and add a custom note
fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"  # Display the region name (NAME_LATN)
        "Applications: %{marker.size}<br>"  # Display the application count
        "(from 2000 to 2023)"  # Custom annotation
    )
)
# Make the map interactive: zoom, pan, hover
fig.update_layout(
    mapbox=dict(
        zoom=6,  # Initial zoom level
        style="carto-positron",  # Map style
        center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    ),
    title="Number of Inventions across Europe",
)

fig.show()

/tmp/ipykernel_1286/1542089585.py:2: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [50]:
# Ensure 'invention_count' is numeric
merged_df_year['invention_count'] = pd.to_numeric(merged_df_year['invention_count'], errors='coerce')

# Drop any rows with NaN values in 'invention_count'
merged_df_year = merged_df_year.dropna(subset=['invention_count'])

# Ensure 'earliest_publn_year' is numeric (if it's not already)
merged_df_year['earliest_publn_year'] = pd.to_numeric(merged_df_year['earliest_publn_year'], errors='coerce')

# Sort by 'earliest_publn_year' to ensure chronological order
merged_df_year = merged_df_year.sort_values(by='earliest_publn_year')

# Plot the centroids (using latitude and longitude) and animate by 'earliest_publn_year'
fig = px.scatter_mapbox(
    merged_df_year,
    lat="latitude",  # Latitude of centroids
    lon="longitude",  # Longitude of centroids
    color="invention_count",  # Color based on application count
    size="invention_count",  # Size of marker based on application count
    color_continuous_scale="Viridis",  # Color scale for better visualisation
    mapbox_style="carto-positron",  # Map style
    animation_frame="earliest_publn_year",  # Animate by 'earliest_publn_year'
    animation_group="NUTS_ID",  # Ensure the same region stays together in each frame
    center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    zoom=6,  # Zoom level
    title="Invention Counts by Region (Centroids) Over Time",  # Add title
    height=600,  # Height of the plot (you can adjust as needed)
    labels={"invention_count": "Number of Inventions"}, 
    hover_name="NAME_LATN",  # Set the place name as the title at the top of hover
    hover_data={
        "latitude": False,  # Exclude latitude
        "longitude": False,  # Exclude longitude
        "invention_count": True,  # Include invention count
        "earliest_publn_year": True  # Include year
    }
)

# Make the map interactive: zoom, pan, hover
fig.update_layout(
    mapbox=dict(
        zoom=6,  # Initial zoom level
        style="carto-positron",  # Map style
        center={"lat": 51.1657, "lon": 10.4515},  # Center map on Germany
    ),
    title="Invention Counts by Region Over Time",
)

# Display the figure
fig.show()

/tmp/ipykernel_1286/3781347528.py:14: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

